<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-03-prompt-engineering/lesson-3.4-context-engineering/notebooks/exercises-3_4.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 3.4: Context Engineering

*Module 3 · Prompt Engineering & Context Design*

> Every LLM has a finite memory. When your conversation, documents, and instructions exceed it, things break silently. This lesson teaches you to manage that memory like a pro.

---

## About this notebook

This notebook contains the **9 runnable code examples** from the published lesson `lesson-3.4.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — The Paradigm Shift — From Prompt Engineering to Context Engineering — `paradigm_shift.py`
2. Step 3 — Token Counting — "You Can't Manage What You Don't Measure" — `part1_token_counter.py`
3. Step 4 — Token Budget Manager — Allocate Space Wisely — `part2_budget_manager.py`
4. Step 5 — Long-Context Reliability — Lost-in-the-Middle & Context Rot — `reliability.py`
5. Step 6 — Conversation Trimming — When History Gets Too Long — `part3_conversation_trimmer.py`
6. Step 7 — Context Compression — Same Information, Fewer Tokens — `part4_compressor.py`
7. Step 8 — Prompt Caching & MCP — Cheap and Dynamic Context — `caching_mcp.py`
8. Step 9 — Summarization Chains — Process Documents Bigger Than the Context — `part5_summarization_chains.py`
9. Step 10 — Project: Complete Context Management Pipeline — `project_context_pipeline.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q google-generativeai


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · The Paradigm Shift — From Prompt Engineering to Context Engineering

Karpathy (2025): "Context engineering is the delicate art and science of filling the context window with just the right information."

**`paradigm_shift.py`** · _Block 1 of 9_


In [ ]:
print("""THE PARADIGM SHIFT (2025):
  OLD: Prompt Engineering = "craft a clever instruction"
  NEW: Context Engineering = "manage ALL information the model sees"

  Karpathy's analogy:  LLM = CPU, Context = RAM, Context Engineering = OS

THE 7-LAYER CONTEXT STACK:
  1. System prompt        2. Few-shot examples
  3. Retrieved docs (RAG) 4. Tool outputs (MCP)
  5. Memory               6. Conversation history
  7. User query (ALWAYS LAST — +30% accuracy boost)

THE FORMULA:
  Available Output = Model Limit - (System + Examples + History
                     + Docs + Tools + Query)
  Every token added to one layer REDUCES all others.

KEY (Anthropic Sep 2025):
  'Most agent failures are context failures, not model failures.'
""")


> **What just happened?** Context engineering (Karpathy/Anthropic 2025) replaces prompt engineering. A prompt is one of 7 layers. Production AI succeeds by managing all layers within a zero-sum token budget. Anthropic: most agent failures are context failures.


### Step 3 · Token Counting — "You Can't Manage What You Don't Measure"

Before managing your context, you need to know how many tokens everything uses.

**`part1_token_counter.py`** · _Block 2 of 9_

TOKEN COUNTER: Know exactly how full you are


In [ ]:
# =============================================
# TOKEN COUNTER: Know exactly how full you are
# =============================================

import google.generativeai as genai
import os

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
model = genai.GenerativeModel("gemini-2.0-flash")

def count_tokens(text: str) -> int:
    """Count tokens in a text string using Gemini's tokenizer."""
    result = model.count_tokens(text)
    return result.total_tokens

# ── Let's measure real content ──
samples = {
    "Short greeting":      "Hello, how are you?",
    "System prompt":       "You are a helpful customer support agent for TechStore India. Always be polite, answer in English or Hindi based on the customer's language, and keep responses under 100 words.",
    "RAG chunk (small)":   "The OnePlus Buds 3 feature 49dB active noise cancellation, 10.4mm dynamic drivers, and up to 44 hours total battery life with the charging case. They support LHDC 5.0 codec for high-resolution audio streaming." * 3,
    "Same in Hindi":       "OnePlus Buds 3 में 49dB एक्टिव नॉइज कैंसिलेशन, 10.4mm डायनामिक ड्राइवर और चार्जिंग केस के साथ कुल 44 घंटे की बैटरी लाइफ है।",
    "Long document":       "Machine learning is a subset of artificial intelligence. " * 200,
}

print("Token counts for different content:\n")
print(f"{'Content':<25} {'Chars':>8} {'Tokens':>8} {'Ratio':>8}")
print("─" * 55)

for label, text in samples.items():
    tokens = count_tokens(text)
    chars = len(text)
    ratio = chars / tokens if tokens > 0 else 0
    print(f"  {label:<25} {chars:>6} {tokens:>6}tk  ~{ratio:.1f} chars/tk")

print("""
Key insights:
  • English averages ~4 characters per token
  • Hindi averages ~2 characters per token (more expensive!)
  • System prompts are usually 100-500 tokens
  • A single RAG chunk is 200-500 tokens
  • 128K context = ~300 pages, BUT that includes EVERYTHING
""")


> **What just happened?** We measured the exact token cost of different content types. The key discovery: Hindi uses roughly 2-3x more tokens than English for the same information. A system prompt costs ~50-100 tokens. A RAG chunk costs ~200-500. These numbers are your "prices" — now you can budget.


### Step 4 · Token Budget Manager — Allocate Space Wisely

Divide your context window into zones: system, history, context, query, and response.

**`part2_budget_manager.py`** · _Block 3 of 9_

TOKEN BUDGET MANAGER — Divides the context window into zones and


In [ ]:
# =============================================
# TOKEN BUDGET MANAGER
# Divides the context window into zones and
# warns you when any zone is overflowing.
# =============================================

class TokenBudget:
    """Manage context window like a financial budget."""
    
    def __init__(self, max_tokens: int = 128000):
        self.max_tokens = max_tokens
        
        # Budget allocation (percentages)
        self.budgets = {
            "system":   int(max_tokens * 0.05),   # 5%  = 6,400 tokens
            "history":  int(max_tokens * 0.25),   # 25% = 32,000 tokens
            "context":  int(max_tokens * 0.30),   # 30% = 38,400 tokens
            "query":    int(max_tokens * 0.05),   # 5%  = 6,400 tokens
            "response": int(max_tokens * 0.35),   # 35% = 44,800 tokens
        }
        
        # Current usage
        self.usage = {k: 0 for k in self.budgets}
    
    def set_usage(self, zone: str, tokens: int):
        """Record how many tokens a zone is using."""
        self.usage[zone] = tokens
    
    def remaining(self, zone: str) -> int:
        """How many tokens are left in a zone?"""
        return max(0, self.budgets[zone] - self.usage[zone])
    
    def total_used(self) -> int:
        return sum(self.usage.values())
    
    def total_remaining(self) -> int:
        return self.max_tokens - self.total_used()
    
    def report(self):
        """Print a visual report of context usage."""
        print(f"\nContext Budget Report ({self.max_tokens:,} tokens)\n")
        print(f"{'Zone':<12} {'Used':>8} {'Budget':>8} {'Left':>8} {'Status':>8}")
        print("─" * 50)
        
        for zone in self.budgets:
            used = self.usage[zone]
            budget = self.budgets[zone]
            left = self.remaining(zone)
            pct = used / budget * 100 if budget > 0 else 0
            
            if pct > 90:   status = "FULL"
            elif pct > 70: status = "HIGH"
            else:          status = "OK"
            
            print(f"  {zone:<12} {used:>6}tk {budget:>6}tk {left:>6}tk  {status}")
        
        print(f"\n  TOTAL: {self.total_used():,} / {self.max_tokens:,} ({self.total_used()/self.max_tokens:.0%})")

# ── Simulate a real chatbot session ──
budget = TokenBudget(max_tokens=128000)

# System prompt: ~200 tokens
budget.set_usage("system", 200)

# After 30 conversation turns: ~15,000 tokens
budget.set_usage("history", 15000)

# RAG retrieved 5 document chunks: ~3,000 tokens
budget.set_usage("context", 3000)

# User's current question: ~50 tokens
budget.set_usage("query", 50)

budget.report()


> **What just happened?** We built a budget manager that tracks token usage across 5 zones. After a simulated chat session (30 turns + RAG), the report shows which zones are filling up. When history hits 90% of its budget, it's time to trim. When context is full, stop adding documents. This prevents the silent failure where the model loses information because the context overflowed.


### Step 5 · Long-Context Reliability — Lost-in-the-Middle & Context Rot

Bigger context ≠ better. Models miss info in the middle and degrade as context grows.

**`reliability.py`** · _Block 4 of 9_


In [ ]:
print("""LOST-IN-THE-MIDDLE (Liu et al. 2023):
  Position 1 (start):   ~75% accuracy
  Position 10 (middle): ~55%  ← 30% DROP!
  Position 20 (end):    ~72%
  FIX: Critical info at START or END. Never the middle.

CONTEXT ROT (Chroma 2025):
  Effective capacity = 50-65% of advertised.
  200K model reliable only to ~130K tokens.
  RULE: Use only 70-80% of the context window.

NEEDLE-IN-HAYSTACK:
  Gemini 1.5 Pro: >99.7% at 1M tokens
  Claude 3 Opus:  <5% drop across 200K
  GPT-4 128K:     Drops after 64K
  Real tasks degrade much more than NIAH suggests.
""")


> **What just happened?** Lost-in-the-middle: 30% accuracy drop for info in the middle. Context rot: effective capacity is 50-65% of advertised. Rules: critical info at start/end, use 70-80% max, focused beats bloated.


### Step 6 · Conversation Trimming — When History Gets Too Long

After 50 turns, your chat history might use 40K tokens. Time to trim smartly.

**`part3_conversation_trimmer.py`** · _Block 5 of 9_

CONVERSATION TRIMMER — 3 strategies to keep history manageable.


In [ ]:
# =============================================
# CONVERSATION TRIMMER
# 3 strategies to keep history manageable.
# =============================================

import google.generativeai as genai

class ConversationTrimmer:
    """Keep conversation history within token budget."""
    
    def __init__(self, max_history_tokens: int = 8000):
        self.max_tokens = max_history_tokens
        self.model = genai.GenerativeModel("gemini-2.0-flash")
    
    def count(self, text):
        return self.model.count_tokens(text).total_tokens
    
    # ── Strategy 1: Sliding Window (simplest) ──
    def sliding_window(self, messages: list[dict]) -> list[dict]:
        """Keep the most recent messages that fit in the budget."""
        kept = []
        total = 0
        
        # Start from the most recent and go backwards
        for msg in reversed(messages):
            msg_tokens = self.count(msg["content"])
            if total + msg_tokens > self.max_tokens:
                break
            kept.insert(0, msg)
            total += msg_tokens
        
        return kept
    
    # ── Strategy 2: Summarize + Recent (best!) ──
    def summarize_and_keep_recent(self, messages: list[dict], keep_recent: int = 6) -> list[dict]:
        """Summarize old messages, keep recent ones in full."""
        
        if len(messages) <= keep_recent:
            return messages  # nothing to trim
        
        # Split: old messages vs recent messages
        old_messages = messages[:-keep_recent]
        recent_messages = messages[-keep_recent:]
        
        # Summarize the old messages
        old_text = "\n".join([f"{m['role']}: {m['content']}" for m in old_messages])
        
        summary_prompt = f"""Summarize this conversation history in 3-4 bullet points.
Keep only the key information: decisions made, questions asked, important facts mentioned.

Conversation:
{old_text}

Summary:"""
        
        summary = self.model.generate_content(summary_prompt).text.strip()
        
        # Build the trimmed history
        summary_message = {
            "role": "system",
            "content": f"[Summary of earlier conversation]\n{summary}"
        }
        
        return [summary_message] + recent_messages

# ── Test with a long conversation ──
trimmer = ConversationTrimmer(max_history_tokens=2000)

# Simulate 20 conversation turns
messages = []
topics = [
    ("Tell me about RAG systems", "RAG combines retrieval with generation..."),
    ("How does the retrieval part work?", "It uses embeddings to find similar..."),
    ("What embedding model should I use?", "For English, text-embedding-004..."),
    ("Can it work with Hindi documents?", "Yes, use multilingual models like..."),
    ("What about chunking strategy?", "Split documents into 500-token chunks..."),
    ("Should chunks overlap?", "Yes, 50-100 token overlap prevents..."),
    ("How many chunks should I retrieve?", "3-5 chunks is the sweet spot..."),
    ("What vector database do you recommend?", "For GCP, use Vertex AI Vector Search..."),
    ("How much does it cost?", "Pricing depends on QPS and storage..."),
    ("Can I use a free alternative?", "ChromaDB is free and runs locally..."),
]

for q, a in topics:
    messages.append({"role": "user", "content": q})
    messages.append({"role": "assistant", "content": a})

print(f"Original: {len(messages)} messages")

# Strategy 1: Sliding window
windowed = trimmer.sliding_window(messages)
print(f"Sliding window: {len(windowed)} messages kept")

# Strategy 2: Summarize + keep recent
summarized = trimmer.summarize_and_keep_recent(messages, keep_recent=6)
print(f"Summarize + recent: {len(summarized)} messages")
print(f"\nSummary of old messages:")
print(f"  {summarized[0]['content'][:200]}...")


> **What just happened?** We built two trimming strategies. Sliding window simply keeps the last N messages that fit — fast but loses all early context. Summarize + recent is smarter: it asks the AI to summarize old messages into 3-4 bullet points, then keeps recent messages in full. The user gets both: memory of early decisions AND full detail of recent conversation. Strategy 2 is what production chatbots use.


### Step 7 · Context Compression — Same Information, Fewer Tokens

Squeeze a 500-token document into 100 tokens without losing the key facts.

**`part4_compressor.py`** · _Block 6 of 9_

CONTEXT COMPRESSOR — 3 levels of compression, each more aggressive.


In [ ]:
# =============================================
# CONTEXT COMPRESSOR
# 3 levels of compression, each more aggressive.
# =============================================

import re

class ContextCompressor:
    """Compress text to fit more info in fewer tokens."""
    
    def __init__(self):
        self.model = genai.GenerativeModel("gemini-2.0-flash", generation_config={"temperature": 0.1})
    
    # ── Level 1: Rule-based (no API call, free!) ──
    def compress_rules(self, text: str) -> str:
        """Remove filler words and compress whitespace."""
        # Remove common filler phrases
        fillers = [
            "it is important to note that", "as mentioned earlier",
            "in order to", "it should be noted that",
            "basically", "essentially", "in other words",
            "as a matter of fact", "for what it's worth",
            "at the end of the day", "the fact that",
        ]
        result = text
        for filler in fillers:
            result = result.replace(filler, "")
        
        # Collapse whitespace
        result = re.sub(r"\s+", " ", result).strip()
        return result
    
    # ── Level 2: Extractive (keep key sentences) ──
    def compress_extractive(self, text: str, target_sentences: int = 3) -> str:
        """Ask the AI to pick the most important sentences."""
        prompt = f"""Extract the {target_sentences} most important sentences from this text.
Return ONLY those sentences, nothing else.

Text: {text}"""
        return self.model.generate_content(prompt).text.strip()
    
    # ── Level 3: Abstractive (rewrite shorter) ──
    def compress_abstractive(self, text: str, target_tokens: int = 100) -> str:
        """Ask the AI to rewrite the text in fewer words."""
        prompt = f"""Rewrite this text in approximately {target_tokens} tokens.
Keep ALL factual information (numbers, names, dates, specifications).
Remove opinions, filler, and repetition.
Use concise language — bullet points are fine.

Text: {text}

Compressed version:"""
        return self.model.generate_content(prompt).text.strip()

# ── Test all 3 levels ──
compressor = ContextCompressor()

long_text = """It is important to note that the OnePlus Buds 3, which were recently 
launched in the Indian market, are essentially a significant upgrade over 
their predecessors. The fact that they feature 49dB active noise cancellation 
is quite impressive for the price point. As a matter of fact, the 10.4mm 
dynamic drivers deliver excellent sound quality with deep bass and clear 
vocals. The battery life is basically outstanding — you get up to 44 hours 
of total playback with the charging case, which is, at the end of the day, 
one of the best in this price segment. In other words, for ₹3,299, these 
earbuds offer exceptional value. It should be noted that they also support 
LHDC 5.0 codec for high-resolution audio streaming, and the IP55 water 
resistance rating makes them suitable for workouts. For what it's worth, 
the only real downside is that the fit might not be ideal for everyone, 
especially during intense physical activities like running or jogging."""

original_tokens = compressor.model.count_tokens(long_text).total_tokens

print(f"Original: {original_tokens} tokens\n")

# Level 1: Rule-based
l1 = compressor.compress_rules(long_text)
l1_tokens = compressor.model.count_tokens(l1).total_tokens
print(f"Level 1 (rules):      {l1_tokens} tokens ({(1-l1_tokens/original_tokens):.0%} reduction)")

# Level 2: Extractive
l2 = compressor.compress_extractive(long_text, target_sentences=3)
l2_tokens = compressor.model.count_tokens(l2).total_tokens
print(f"Level 2 (extractive): {l2_tokens} tokens ({(1-l2_tokens/original_tokens):.0%} reduction)")

# Level 3: Abstractive
l3 = compressor.compress_abstractive(long_text, target_tokens=60)
l3_tokens = compressor.model.count_tokens(l3).total_tokens
print(f"Level 3 (abstractive):{l3_tokens} tokens ({(1-l3_tokens/original_tokens):.0%} reduction)")

print(f"\nCompressed version (Level 3):\n  {l3}")


> **What just happened?** Three compression levels: Rules (free, ~20% reduction — removes filler words), Extractive (1 API call, ~50% — keeps only the most important sentences), Abstractive (1 API call, ~70% — rewrites everything shorter while keeping all facts). For the OnePlus review: 170 tokens → 50 tokens with all specs preserved. Use Level 1 always (it's free), Level 2-3 when you're running low on budget.


### Step 8 · Prompt Caching & MCP — Cheap and Dynamic Context

Caching: 90% off repeated tokens. MCP: standardized dynamic context injection.

**`caching_mcp.py`** · _Block 7 of 9_


In [ ]:
print("""PROMPT CACHING:
  Anthropic: 90% off | OpenAI: 50-90% off | Google: 75-90% off
  Architecture: Static FIRST (cached), dynamic LAST (fresh)
  Cached: system + tools + examples + policies
  Fresh:  history + docs + user query
  Impact: 85% less latency, 70% less cost

MCP (Model Context Protocol):
  Anthropic Nov 2024, adopted by all providers by mid-2025.
  Three primitives: Resources, Tools, Prompts
  97M+ monthly SDK downloads.
  
  JUST-IN-TIME CONTEXT (best practice):
  ✗ Pre-load everything (context stuffing)
  ✓ Carry IDs, retrieve when needed
  Tool definitions consume tokens EVERY call — load on demand.
  For Hindi at 3x token cost: caching is survival.
""")


> **What just happened?** Caching: 90% off for repeated context. Static first, dynamic last. MCP: Resources + Tools + Prompts for just-in-time context. Tool defs cost tokens every call — load on demand. Essential for multilingual apps.


### Step 9 · Summarization Chains — Process Documents Bigger Than the Context

What if your document is 500 pages but the context window is only 300?

**`part5_summarization_chains.py`** · _Block 8 of 9_

SUMMARIZATION CHAINS — Process documents MUCH longer than the


In [ ]:
# =============================================
# SUMMARIZATION CHAINS
# Process documents MUCH longer than the
# context window. Two strategies.
# =============================================

class SummarizationChain:
    """Summarize documents of ANY length using chain strategies."""
    
    def __init__(self, chunk_size: int = 3000):
        self.chunk_size = chunk_size  # characters per chunk
        self.model = genai.GenerativeModel("gemini-2.0-flash")
    
    def _chunk(self, text: str) -> list[str]:
        """Split text into chunks with overlap."""
        chunks = []
        start = 0
        while start < len(text):
            end = start + self.chunk_size
            chunk = text[start:end]
            chunks.append(chunk)
            start = end - 200  # 200 char overlap
        return chunks
    
    def _summarize_one(self, text: str, instruction: str = "") -> str:
        prompt = f"""{instruction}
Summarize the following text in 3-5 concise bullet points.
Keep all important facts, numbers, and names.

Text: {text}

Summary:"""
        return self.model.generate_content(prompt).text.strip()
    
    # ── Strategy 1: Map-Reduce ──
    def map_reduce(self, text: str) -> dict:
        """
        MAP:    Summarize each chunk independently
        REDUCE: Combine all summaries into one final summary
        """
        chunks = self._chunk(text)
        print(f"  Split into {len(chunks)} chunks")
        
        # MAP: summarize each chunk
        chunk_summaries = []
        for i, chunk in enumerate(chunks):
            summary = self._summarize_one(chunk)
            chunk_summaries.append(summary)
            print(f"  Chunk {i+1}/{len(chunks)} summarized")
        
        # REDUCE: combine summaries
        all_summaries = "\n\n".join([f"Section {i+1}:\n{s}" for i, s in enumerate(chunk_summaries)])
        
        final = self._summarize_one(all_summaries, "Create a comprehensive final summary combining all sections.")
        
        return {"final_summary": final, "chunk_summaries": chunk_summaries, "num_chunks": len(chunks)}
    
    # ── Strategy 2: Refine ──
    def refine(self, text: str) -> dict:
        """
        Start with chunk 1 summary.
        For each next chunk, REFINE the summary with new info.
        """
        chunks = self._chunk(text)
        print(f"  Split into {len(chunks)} chunks")
        
        # Start with first chunk
        running_summary = self._summarize_one(chunks[0])
        print(f"  Chunk 1/{len(chunks)}: initial summary created")
        
        # Refine with each subsequent chunk
        for i in range(1, len(chunks)):
            refine_prompt = f"""Here is an existing summary of a document:
{running_summary}

Here is new content from the next section:
{chunks[i]}

Update the summary to include important new information from this section.
Keep the summary concise (5-8 bullet points total).

Updated summary:"""
            
            running_summary = self.model.generate_content(refine_prompt).text.strip()
            print(f"  Chunk {i+1}/{len(chunks)}: summary refined")
        
        return {"final_summary": running_summary, "num_chunks": len(chunks)}

# ── Test with a long document ──
chain = SummarizationChain(chunk_size=2000)

# Simulate a long document (in real use, load from a file)
long_document = """
Artificial Intelligence in Indian Agriculture: A Comprehensive Report

Chapter 1: Current State of Indian Agriculture
India's agricultural sector employs over 42% of the workforce and contributes 
approximately 18% to the GDP. The sector faces challenges including unpredictable 
monsoons, pest infestations, soil degradation, and fragmented land holdings. 
The average farm size in India is just 1.08 hectares, making it difficult for 
individual farmers to invest in technology...
""" * 10  # Make it long enough to need chunking

print(f"Document length: {len(long_document)} characters\n")

print("Strategy 1: Map-Reduce")
result1 = chain.map_reduce(long_document)
print(f"\nFinal summary:\n{result1['final_summary']}\n")

print("Strategy 2: Refine")
result2 = chain.refine(long_document)
print(f"\nFinal summary:\n{result2['final_summary']}")


> **What just happened?** We built two ways to summarize documents that don't fit in the context window: Map-Reduce (summarize each chunk independently, then combine — fast, can run in parallel) and Refine (start with chunk 1, update the summary as each new chunk arrives — slower but preserves connections between sections). Both can process documents of unlimited length by breaking them into manageable chunks.


### Step 10 · Project: Complete Context Management Pipeline

Combine everything into a production-ready chatbot context manager.

**`project_context_pipeline.py`** · _Block 9 of 9_

COMPLETE CONTEXT MANAGEMENT PIPELINE — Combines: budget + trimming + compression


In [ ]:
# =============================================
# COMPLETE CONTEXT MANAGEMENT PIPELINE
# Combines: budget + trimming + compression
# into one production-ready system.
# =============================================

class ContextPipeline:
    """Full context management for a production chatbot."""
    
    def __init__(self, max_tokens=128000, system_prompt=""):
        self.budget = TokenBudget(max_tokens)
        self.trimmer = ConversationTrimmer(max_history_tokens=int(max_tokens * 0.25))
        self.compressor = ContextCompressor()
        self.system_prompt = system_prompt
        self.messages = []
        self.model = genai.GenerativeModel("gemini-2.0-flash", system_instruction=system_prompt)
    
    def _count(self, text):
        return self.model.count_tokens(text).total_tokens
    
    def add_message(self, role: str, content: str):
        self.messages.append({"role": role, "content": content})
    
    def build_context(self, user_query: str, rag_chunks: list[str] = None) -> str:
        """Build the optimal context within budget."""
        
        # 1. Track system prompt tokens
        self.budget.set_usage("system", self._count(self.system_prompt))
        
        # 2. Track query tokens
        self.budget.set_usage("query", self._count(user_query))
        
        # 3. Compress and fit RAG context
        context_text = ""
        if rag_chunks:
            # Compress each chunk if needed
            remaining = self.budget.budgets["context"]
            for chunk in rag_chunks:
                chunk_tokens = self._count(chunk)
                if chunk_tokens > remaining:
                    # Compress to fit!
                    chunk = self.compressor.compress_rules(chunk)
                    chunk_tokens = self._count(chunk)
                if chunk_tokens <= remaining:
                    context_text += chunk + "\n\n"
                    remaining -= chunk_tokens
            self.budget.set_usage("context", self._count(context_text))
        
        # 4. Trim conversation history to fit
        history_budget = self.budget.budgets["history"]
        total_history_tokens = sum(self._count(m["content"]) for m in self.messages)
        
        if total_history_tokens > history_budget:
            print(f"  History exceeds budget ({total_history_tokens} > {history_budget}). Trimming...")
            self.messages = self.trimmer.summarize_and_keep_recent(self.messages, keep_recent=6)
        
        self.budget.set_usage("history", sum(self._count(m["content"]) for m in self.messages))
        
        # 5. Assemble the final prompt
        history_text = "\n".join([f"{m['role']}: {m['content']}" for m in self.messages])
        
        full_prompt = f"""Previous conversation:
{history_text}

{"Relevant context:" + chr(10) + context_text if context_text else ""}

User: {user_query}
Assistant:"""
        
        return full_prompt
    
    def chat(self, user_query: str, rag_chunks=None) -> str:
        """Send a message and get a response with full context management."""
        self.add_message("user", user_query)
        
        prompt = self.build_context(user_query, rag_chunks)
        response = self.model.generate_content(prompt).text.strip()
        
        self.add_message("assistant", response)
        return response

# ── Run it! ──
pipeline = ContextPipeline(
    max_tokens=128000,
    system_prompt="You are a helpful AI tutor for the Netsetos GenAI course. Answer student questions clearly and concisely."
)

# Simulate a conversation
questions = [
    "What is RAG?",
    "How does the retrieval part work?",
    "What embedding model should I use?",
    "Can you explain the chunking strategy?",
]

for q in questions:
    print(f"\nStudent: {q}")
    answer = pipeline.chat(q)
    print(f"Tutor: {answer[:150]}...")

# Show budget status
pipeline.budget.report()


> **What just happened?** We combined all 4 techniques into one ContextPipeline class: (1) tracks token budget across zones, (2) compresses RAG chunks if they don't fit, (3) trims conversation history when it exceeds budget, (4) assembles the optimal prompt. This is the exact pattern production chatbots use — every turn, the system decides what to keep, what to compress, and what to drop, all within the token budget.


---

## Wrap-up

You've walked through all 9 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
